# BoltzGen Nanobody Scoring
**Target:** P05231 (IL-6)  
**Source:** [github.com/schneider-weave/boltz_scoring](https://github.com/schneider-weave/boltz_scoring)

Pipeline: **folding → analysis → filtering** (inverse folding skipped — sequences are fixed)

> ⚠️ **Kaggle settings required:** Accelerator = GPU (T4 or P100), Internet = ON

## 1. Clone repo & install

In [ ]:
import os

REPO_URL = "https://github.com/schneider-weave/boltz_scoring"
REPO_DIR = "/kaggle/working/boltz_scoring"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

print("Repo ready:", REPO_DIR)

In [ ]:
%%bash
# CUDA-compatible torch for Kaggle (CUDA 12.x)
pip install -q torch --index-url https://download.pytorch.org/whl/cu121

# cuequivariance wheels required by boltzgen
pip install -q \
    cuequivariance-ops-torch-cu12 \
    cuequivariance-torch \
    cuequivariance-ops-cu12

# Install boltzgen from local source (bittensor-free)
pip install -q -e /kaggle/working/boltz_scoring/boltzgen

echo "Done"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!boltzgen -v

## 2. Paths & configuration

In [ ]:
from pathlib import Path

REPO_DIR       = Path("/kaggle/working/boltz_scoring")
SCORING_INPUTS = REPO_DIR / "scoring_inputs"   # 227 per-nanobody YAML files
OUTPUT_DIR     = Path("/kaggle/working/scoring_results")
CACHE_DIR      = Path("/kaggle/working/cache")  # model weights ~6 GB

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

yaml_files = sorted(SCORING_INPUTS.glob("*.yaml"))
print(f"Input YAMLs : {len(yaml_files)}")
print(f"MSA exists  : {(REPO_DIR / 'msa_files/P05231.a3m').exists()}")
print(f"Output dir  : {OUTPUT_DIR}")

# Preview one input file
print(f"\nSample input ({yaml_files[0].name}):")
print(yaml_files[0].read_text())

## 3. Download model weights (~6 GB, cached after first run)

In [ ]:
!boltzgen download all --cache {CACHE_DIR}

## 4. Step 1 — Folding

Folds each nanobody+target complex. Writes `.cif` and `.npz` files into `scoring_results/`.

With `--skip_inverse_folding` the pipeline reads sequences directly from the YAML,
skips the design + inverse-fold steps, and runs **folding → analysis → filtering**.

In [ ]:
import subprocess, sys

fold_cmd = [
    "boltzgen", "run", str(SCORING_INPUTS),
    "--output",           str(OUTPUT_DIR),
    "--protocol",         "nanobody-anything",
    "--cache",            str(CACHE_DIR),
    "--skip_inverse_folding",
    "--num_designs",      "1",
    "--steps",            "folding",
    "--no_subprocess",
]

print("Running:", " ".join(fold_cmd))
print("-" * 60)
result = subprocess.run(fold_cmd)
print("\n[Folding exit code]", result.returncode)

In [ ]:
# Verify folding produced both .cif and .npz files
from boltzgen.data import const

cif_files = sorted(OUTPUT_DIR.glob("*.cif"))
npz_dir   = OUTPUT_DIR / const.folding_dirname   # fold_out_npz/
npz_files = sorted(npz_dir.glob("*.npz")) if npz_dir.exists() else []

print(f"CIF files   : {len(cif_files)}")
print(f"NPZ dir     : {npz_dir}  (exists={npz_dir.exists()})")
print(f"NPZ files   : {len(npz_files)}")

if len(npz_files) == 0:
    print("\n[WARNING] No NPZ files found — listing output dir:")
    for p in sorted(OUTPUT_DIR.rglob("*"))[:30]:
        print(" ", p)
else:
    print("\n[OK] Folding output looks correct")

## 5. Step 2 — Analysis

Computes binding metrics from the folded structures: `delta_sasa`, `plip_hbonds`, `plddt`, `iptm`, etc.  
Writes `aggregate_metrics_analyze.csv` into `scoring_results/`.

In [ ]:
analysis_cmd = [
    "boltzgen", "run", str(SCORING_INPUTS),
    "--output",           str(OUTPUT_DIR),
    "--protocol",         "nanobody-anything",
    "--cache",            str(CACHE_DIR),
    "--skip_inverse_folding",
    "--num_designs",      "1",
    "--steps",            "analysis",
    "--no_subprocess",
]

print("Running:", " ".join(analysis_cmd))
print("-" * 60)
result = subprocess.run(analysis_cmd)
print("\n[Analysis exit code]", result.returncode)

In [ ]:
# Locate aggregate_metrics_analyze.csv — it's written directly into design_dir
# which equals OUTPUT_DIR when --skip_inverse_folding is used
candidates = [
    OUTPUT_DIR / "aggregate_metrics_analyze.csv",
    OUTPUT_DIR / "intermediate_designs" / "aggregate_metrics_analyze.csv",
    OUTPUT_DIR / "intermediate_designs_inverse_folded" / "aggregate_metrics_analyze.csv",
]

metrics_csv = next((p for p in candidates if p.exists()), None)

if metrics_csv is None:
    # Fallback: recursive search
    found = list(OUTPUT_DIR.rglob("aggregate_metrics_analyze.csv"))
    if found:
        metrics_csv = found[0]

if metrics_csv is None:
    print("[ERROR] aggregate_metrics_analyze.csv not found. Output dir contents:")
    for p in sorted(OUTPUT_DIR.rglob("*"))[:40]:
        print(" ", p)
else:
    print(f"[OK] Metrics CSV found: {metrics_csv}")

## 6. Step 3 — Filtering (ranking)

Fast step (~20 s). Ranks designs by quality score and writes `final_ranked_designs/`.

In [ ]:
filter_cmd = [
    "boltzgen", "run", str(SCORING_INPUTS),
    "--output",           str(OUTPUT_DIR),
    "--protocol",         "nanobody-anything",
    "--cache",            str(CACHE_DIR),
    "--skip_inverse_folding",
    "--num_designs",      "1",
    "--steps",            "filtering",
    "--budget",           "20",   # top-20 final set
    "--no_subprocess",
]

print("Running:", " ".join(filter_cmd))
print("-" * 60)
result = subprocess.run(filter_cmd)
print("\n[Filtering exit code]", result.returncode)

## 7. Load & display results

In [ ]:
import pandas as pd

df = pd.read_csv(metrics_csv)
print(f"Loaded {len(df)} results — columns: {list(df.columns)}")
df.head(3)

In [ ]:
DESIRED_METRICS = [
    "id",
    "delta_sasa_refolded",          # buried surface area      (↑ better)
    "plip_hbonds_refolded",         # H-bonds at interface     (↑ better)
    "noncovalents_refolded",        # non-covalent contacts    (↑ better)
    "largest_hydrophobic_refolded", # hydrophobic patch size   (↓ better)
    "plddt",                        # folding confidence       (↑ better)
    "ptm",                          # predicted TM-score       (↑ better)
    "iptm",                         # interface pTM            (↑ better)
]

available = [c for c in DESIRED_METRICS if c in df.columns]
missing   = [c for c in DESIRED_METRICS if c not in df.columns]
if missing:
    print(f"Not available: {missing}")

df_scores = df[available].copy()
df_scores["nanobody_id"]   = df_scores["id"].str.extract(r"(design_spec_\d+)", expand=False)
df_scores["original_rank"] = df_scores["id"].str.extract(r"rank=(\d+)", expand=False).astype(float)

# Rank: primarily by delta_sasa, secondarily by hbonds, then iptm
sort_cols = [c for c in ["delta_sasa_refolded", "plip_hbonds_refolded", "iptm"] if c in df_scores.columns]
df_ranked = df_scores.sort_values(sort_cols, ascending=False).reset_index(drop=True)
df_ranked.index += 1
df_ranked.index.name = "boltzgen_rank"

print(f"\nTop 20 nanobodies (ranked by {sort_cols}):")
df_ranked.head(20)

## 8. Visualize

In [ ]:
import matplotlib.pyplot as plt

plot_metrics = [c for c in [
    "delta_sasa_refolded", "plip_hbonds_refolded",
    "largest_hydrophobic_refolded", "plddt", "iptm",
] if c in df_ranked.columns]

fig, axes = plt.subplots(1, len(plot_metrics), figsize=(4 * len(plot_metrics), 4))
if len(plot_metrics) == 1:
    axes = [axes]

for ax, metric in zip(axes, plot_metrics):
    ax.hist(df_ranked[metric].dropna(), bins=25, edgecolor="white", color="steelblue")
    ax.set_title(metric.replace("_", "\n"), fontsize=8)
    ax.set_xlabel("score")

plt.suptitle("BoltzGen Score Distributions — P05231 Nanobodies", fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig("/kaggle/working/score_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
x_col, y_col = "delta_sasa_refolded", "plip_hbonds_refolded"

if x_col in df_ranked.columns and y_col in df_ranked.columns:
    fig, ax = plt.subplots(figsize=(7, 5))
    sc = ax.scatter(df_ranked[x_col], df_ranked[y_col],
                    c=df_ranked.index, cmap="viridis_r", alpha=0.8, s=40)
    plt.colorbar(sc, ax=ax, label="boltzgen_rank")
    ax.set_xlabel("delta_sasa_refolded  (↑ better)")
    ax.set_ylabel("plip_hbonds_refolded  (↑ better)")
    ax.set_title("Binding Quality — P05231 Nanobodies")
    for i, row in df_ranked.head(5).iterrows():
        ax.annotate(str(row.get("nanobody_id", i)), (row[x_col], row[y_col]),
                    fontsize=7, xytext=(4, 4), textcoords="offset points")
    plt.tight_layout()
    plt.savefig("/kaggle/working/sasa_vs_hbonds.png", dpi=150, bbox_inches="tight")
    plt.show()

## 9. Save results

In [ ]:
out_csv = "/kaggle/working/nanobody_boltzgen_scores.csv"
df_ranked.to_csv(out_csv)
print(f"Saved: {out_csv}")
print("\nSummary:")
df_ranked[plot_metrics].describe().round(3)

In [ ]:
# Attach original sequences from input YAMLs
import yaml as pyyaml

seq_map = {}
for yf in sorted(SCORING_INPUTS.glob("*.yaml")):
    data = pyyaml.safe_load(yf.read_text())
    for entity in data.get("entities", []):
        if entity.get("protein", {}).get("id") == "B":
            seq_map[yf.stem] = entity["protein"]["sequence"]

df_ranked["sequence"] = df_ranked["nanobody_id"].map(
    lambda x: next((v for k, v in seq_map.items() if x and x in k), None)
)

# Save top 10 as FASTA
fasta_out = "/kaggle/working/top10_boltzgen_scored.fasta"
with open(fasta_out, "w") as f:
    for rank, row in df_ranked.head(10).iterrows():
        nb_id = row.get("nanobody_id") or f"nb_{rank}"
        sasa  = f"{row['delta_sasa_refolded']:.2f}" if "delta_sasa_refolded" in row else "NA"
        hb    = row.get("plip_hbonds_refolded", "NA")
        seq   = row.get("sequence") or ""
        f.write(f">{nb_id}|boltzgen_rank={rank}|delta_sasa={sasa}|hbonds={hb}\n{seq}\n")

print(f"Saved: {fasta_out}")
print(open(fasta_out).read())

## Output files

| File | Description |
|---|---|
| `nanobody_boltzgen_scores.csv` | Full ranked table with all metrics |
| `top10_boltzgen_scored.fasta` | Top 10 sequences with scores in FASTA header |
| `score_distributions.png` | Histogram of each scoring metric |
| `sasa_vs_hbonds.png` | Scatter: buried surface vs H-bonds |
| `scoring_results/` | Raw boltzgen output (CIFs, NPZs, CSVs) |

## Key metrics

| Metric | Direction | Meaning |
|---|---|---|
| `delta_sasa_refolded` | ↑ higher = better | Buried surface area at the interface |
| `plip_hbonds_refolded` | ↑ higher = better | Number of H-bonds between nanobody and target |
| `largest_hydrophobic_refolded` | ↓ lower = better | Hydrophobic patch size (developability risk) |
| `plddt` | ↑ higher = better | Per-residue folding confidence |
| `iptm` | ↑ higher = better | Interface predicted TM-score |